In [3]:
# ============================== CONFIGURACIÓN ==============================
from pathlib import Path

# Carpeta donde viven los scripts de la suite (14_*.py y 13_*.R)
DIR_SUITE = Path(r"C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\Diagnosticos_python")

# Modo de ejecución: "fixture" (prueba sin R) o "real" (datos de la tesis)
MODO = "real"

# ---- modo real -------------------------------------------------------------
CADENAS_REAL  = [Path(r"C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\mcmc_salida_m2")]     # una o más carpetas del Gibbs
DIAG_REAL     = Path(r"C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\diag_m2_py")           # carpeta de resultados
COMPARAR_REAL = None                         # p. ej. Path("mcmc_salida_m1")
NOMBRES_REAL  = ["t-Student (M2)", "Normal (M1)"]

# ---- modo fixture ----------------------------------------------------------
DIR_FIXTURE  = Path("fixture_m2")
DIAG_FIXTURE = Path("diag_fixture")

# ---- opciones de la corrida ------------------------------------------------
RAPIDO  = False     # None = automático (True en fixture, False en real)
SIN_LOO = False    # True para omitir PSIS-LOO (ahorra tiempo con N grande)
DRAWS   = None     # draws del PPC; None = automático (120 rápido / 400 normal)
SEED    = 0
SIN_PDF = False    # True para no generar INFORME_M2.pdf
MOSTRAR_TODAS_LAS_FIGURAS = False   # False = solo las figuras clave al final

# ---- resolución del modo (no editar) ---------------------------------------
if MODO == "fixture":
    CADENAS  = [DIR_FIXTURE / "m2" / "cad1", DIR_FIXTURE / "m2" / "cad2"]
    DIR_DIAG = DIAG_FIXTURE
    COMPARAR = DIR_FIXTURE / "rival_peor" / "cad1"
    NOMBRES  = ["t-Student (M2)", "rival degradado"]
    RAPIDO   = True if RAPIDO is None else RAPIDO
elif MODO == "real":
    CADENAS  = [Path(p) for p in CADENAS_REAL]
    DIR_DIAG = Path(DIAG_REAL)
    COMPARAR = Path(COMPARAR_REAL) if COMPARAR_REAL else None
    NOMBRES  = NOMBRES_REAL
    RAPIDO   = False if RAPIDO is None else RAPIDO
else:
    raise ValueError("MODO debe ser 'fixture' o 'real'")
DRAWS_EF = DRAWS if DRAWS is not None else (120 if RAPIDO else 400)

print(f"MODO      : {MODO}{'  (rápido)' if RAPIDO else ''}")
print(f"Cadenas   : {[str(p) for p in CADENAS]}")
print(f"Diagnóst. : {DIR_DIAG}")
print(f"Comparar  : {COMPARAR if COMPARAR else '(sin comparación)'}")
print(f"Draws PPC : {DRAWS_EF} | PSIS-LOO: {'omitido' if (SIN_LOO or RAPIDO) else 'sí'}")

MODO      : real
Cadenas   : ['C:\\Users\\USUARIO\\OneDrive - Universidad Nacional de Colombia\\Tesis\\Escritorio\\Códigos\\Modelos\\Modelo 2\\mcmc_salida_m2']
Diagnóst. : C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\diag_m2_py
Comparar  : (sin comparación)
Draws PPC : 400 | PSIS-LOO: sí


In [4]:
# ===================== CARGA DE LA SUITE Y VERIFICACIÓN =====================
import importlib.util, sys, json
import numpy as np, pandas as pd, matplotlib, scipy

def cargar(nombre):
    """Importa un script de la suite por ruta (los nombres empiezan con dígito)."""
    ruta = DIR_SUITE / nombre
    if not ruta.exists():
        raise FileNotFoundError(f"No encuentro {ruta}: ajustar DIR_SUITE.")
    spec = importlib.util.spec_from_file_location(
        nombre.replace(".py", "").replace(".", "_"), ruta)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def _chk(cond, texto):
    print(("  [OK]    " if cond else "  [FALTA] ") + texto)
    return bool(cond)

def preflight():
    print(f"numpy {np.__version__} | scipy {scipy.__version__} | "
          f"pandas {pd.__version__} | matplotlib {matplotlib.__version__}")
    print("Scripts de la suite:")
    ok = True
    for s in ["14_0_utilidades_M2.py", "14_a_convergencia_M2.py",
              "14_b_posterior_M2.py", "14_c_ajuste_M2.py",
              "14_d_ppc_M2.py", "14_e_prueba_sintetica_M2.py",
              "14_f_informe_M2.py"]:
        ok &= _chk((DIR_SUITE / s).exists(), s)
    _chk((DIR_SUITE / "13_exportar_python_M2.R").exists(),
         "13_exportar_python_M2.R (solo se usa desde R)")
    if MODO == "fixture" and not (CADENAS[0] / "export_python" / "control.bin").exists():
        print("  [INFO]  el fixture aún no existe: generarlo en la celda siguiente.")
        return ok
    print("Cadenas:")
    for d in CADENAS + ([COMPARAR] if COMPARAR else []):
        ex = d / "export_python"
        c = ex / "control.bin"
        base = _chk(c.exists(), f"{d}  export_python/ (Etapa 13)")
        if c.exists():
            ctl = np.fromfile(c, dtype="<f8")
            base &= _chk(len(ctl) == 2 and abs(ctl[0] - np.pi) < 1e-9,
                         f"{d}  control.bin = (pi, n_save={int(ctl[1]) if len(ctl)==2 else '?'})")
        _chk((d / "loglik.bin").exists(), f"{d}  loglik.bin (WAIC/LOO y comparación)")
        _chk((ex / "y_vec.bin").exists(), f"{d}  y_vec/X_star (PPC, DIC y capa xi)")
        if (ex / "holdout").exists():
            print(f"  [OK]    {d}  holdout/ (capa fuera de muestra habilitada)")
        ok &= base
    if not ok and MODO == "real":
        print("Falta la Etapa 13 en alguna carpeta: ver la sección de R más abajo.")
    return ok

PREFLIGHT_OK = preflight()

numpy 1.26.4 | scipy 1.12.0 | pandas 2.1.4 | matplotlib 3.8.0
Scripts de la suite:
  [OK]    14_0_utilidades_M2.py
  [OK]    14_a_convergencia_M2.py
  [OK]    14_b_posterior_M2.py
  [OK]    14_c_ajuste_M2.py
  [OK]    14_d_ppc_M2.py
  [OK]    14_e_prueba_sintetica_M2.py
  [OK]    14_f_informe_M2.py
  [OK]    13_exportar_python_M2.R (solo se usa desde R)
Cadenas:
  [OK]    C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\mcmc_salida_m2  export_python/ (Etapa 13)
  [OK]    C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\mcmc_salida_m2  control.bin = (pi, n_save=6000)
  [OK]    C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\mcmc_salida_m2  loglik.bin (WAIC/LOO y comparación)
  [OK]    C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\mcmc_salida_m2  y_vec/X_star (PPC, DIC y 

In [5]:
if MODO == "fixture":
    if (CADENAS[0] / "export_python" / "control.bin").exists():
        print(f"El fixture ya existe en {DIR_FIXTURE}; se reutiliza.")
        print("(borrar la carpeta y volver a correr esta celda para regenerarlo)")
    else:
        _argv = sys.argv
        sys.argv = ["14_e", str(DIR_FIXTURE)]
        try:
            cargar("14_e_prueba_sintetica_M2.py").main()
        finally:
            sys.argv = _argv
else:
    print("MODO real: esta celda no aplica.")

MODO real: esta celda no aplica.


In [6]:
import time

argv = [str(p) for p in CADENAS] + ["--diag", str(DIR_DIAG), "--seed", str(SEED)]
if DRAWS is not None:
    argv += ["--draws", str(DRAWS)]
if RAPIDO:
    argv += ["--rapido"]
if SIN_LOO:
    argv += ["--sin-loo"]
if SIN_PDF:
    argv += ["--sin-pdf"]
if COMPARAR is not None:
    argv += ["--comparar", str(COMPARAR), "--nombres", *NOMBRES]

print("argv:", " ".join(argv), "\n")
_t0 = time.time()
rc = cargar("14_f_informe_M2.py").main(argv)
print(f"\nTerminado en {time.time() - _t0:.0f} s | código de salida: {rc}")

argv: C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\mcmc_salida_m2 --diag C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\diag_m2_py --seed 0 

INFORME CONSOLIDADO - Modelo 2 (14f)
Cadenas: ['C:\\Users\\USUARIO\\OneDrive - Universidad Nacional de Colombia\\Tesis\\Escritorio\\Códigos\\Modelos\\Modelo 2\\mcmc_salida_m2'] | diag: C:\Users\USUARIO\OneDrive - Universidad Nacional de Colombia\Tesis\Escritorio\Códigos\Modelos\Modelo 2\diag_m2_py | draws PPC: 400

########################################################################
# 14a convergencia
########################################################################
DIAGNOSTICO DE CONVERGENCIA - Modelo 2 (14a, Python)
Cadenas: 1 | muestras/cadena (min): 6000 | N=499983, K=33, n_cells=1107 | p_E=63, p_M=11, p_D=6
NOTA: con 1 sola cadena, Rhat es el split-Rhat. Para el Gelman-Rubin real, correr 2-4 cadenas con semillas distin

## Ejecución por etapas

Estas celdas repiten una sola pieza sobre la misma carpeta de diagnóstico, útil por ejemplo para regenerar el PPC con más draws o rehacer la comparación con otro rival sin recorrer todo. La corrida completa de arriba ya las incluye; después de repetir alguna, volver a correr la celda 14f si se quiere refrescar el resumen y el PDF consolidado.

In [ ]:
# --- 14a: convergencia ---
argv = [str(p) for p in CADENAS] + ["--diag", str(DIR_DIAG)]
if RAPIDO:
    argv += ["--sin-kappa2-jk"]
cargar("14_a_convergencia_M2.py").main(argv)

In [ ]:
# --- 14b: posterior y robustez xi ---
argv = [str(p) for p in CADENAS] + ["--diag", str(DIR_DIAG),
                                    "--max-draws-xi", str(min(DRAWS_EF, 400))]
cargar("14_b_posterior_M2.py").main(argv)

In [ ]:
# --- 14c: WAIC / PSIS-LOO / DIC (+ comparación) ---
argv = [str(CADENAS[0]), "--diag", str(DIR_DIAG)]
if SIN_LOO or RAPIDO:
    argv += ["--sin-loo"]
if COMPARAR is not None:
    argv += ["--comparar", str(COMPARAR), "--nombres", *NOMBRES]
cargar("14_c_ajuste_M2.py").main(argv)

In [ ]:
# --- 14d: chequeo predictivo posterior (+ hold-out) ---
argv = [str(p) for p in CADENAS] + ["--diag", str(DIR_DIAG),
                                    "--draws", str(DRAWS_EF), "--seed", str(SEED)]
cargar("14_d_ppc_M2.py").main(argv)

## Resultados

In [ ]:
# --- resumen en prosa ---
_f = DIR_DIAG / "00_RESUMEN_M2.txt"
print(_f.read_text(encoding="utf-8") if _f.exists()
      else "Aún no existe 00_RESUMEN_M2.txt: correr la celda 14f.")

In [ ]:
# --- tablas clave ---
from IPython.display import display

def _csv(nombre):
    f = DIR_DIAG / nombre
    return pd.read_csv(f) if f.exists() else None

comp = DIR_DIAG / "comparacion_resumen_M2.json"
if comp.exists():
    c = json.loads(comp.read_text(encoding="utf-8"))
    veredicto = ("diferencia creíble (>2 SE)" if c["creible_2se"]
                 else "no concluyente (<2 SE)")
    print(f"Comparación ({c['criterio']}): Δelpd({c['nombre_A']} − {c['nombre_B']}) "
          f"= {c['delta_elpd']:+.1f} (SE {c['se']:.1f}) → preferido "
          f"{c['mejor']}, {veredicto}.\n")

t = _csv("ajuste_M2.csv")
if t is not None:
    print("Criterios de información:")
    display(t.T.rename(columns={0: "valor"}).round(2))

t = _csv("resumen_convergencia_M2.csv")
if t is not None:
    print("Peores parámetros por Rhat:")
    display(t.sort_values("rhat", ascending=False)
             .head(8)[["bloque", "parametro", "rhat", "ess_bulk",
                       "ess_tail", "mcse"]].round(4))

t = _csv("resumen_coeficientes_M2.csv")
if t is not None:
    s = t[t["senal_ic95"] == "*"]
    print(f"Coeficientes con IC95% que excluye el cero ({len(s)} de {len(t)}):")
    display(s.reindex(s["mediana"].abs().sort_values(ascending=False).index)
             .head(12)[["nivel", "parametro", "mediana", "q025", "q975"]]
             .round(3))

t = _csv("ppc_cobertura_M2.csv")
if t is not None:
    print("Cobertura de intervalos predictivos:")
    display(t[t["nivel"] == "GLOBAL"].round(3))

t = _csv("ppc_estadisticos_M2.csv")
if t is not None:
    print("Estadísticos de test del PPC (bandera = p_bayes extremo):")
    display(t[["estadistico", "T_obs", "T_rep_medio", "p_bayes",
               "bandera"]].round(3))

In [ ]:
# --- figuras principales (o todas, según MOSTRAR_TODAS_LAS_FIGURAS) ---
from IPython.display import Image, display

CLAVE = ["fig_14a_02_nu.png", "fig_14a_04_rhat_ess.png",
         "fig_14b_01_forest_por_nivel.png", "fig_14b_07_robustez_xi.png",
         "fig_14c_03_comparacion.png", "fig_14d_01_densidad_overlay.png",
         "fig_14d_02_pit.png", "fig_14d_03_qq_calibracion.png"]

figs = (sorted(DIR_DIAG.glob("fig_14*.png")) if MOSTRAR_TODAS_LAS_FIGURAS
        else [DIR_DIAG / f for f in CLAVE if (DIR_DIAG / f).exists()])
if not figs:
    print("Sin figuras todavía: correr la celda 14f.")
for f in figs:
    print("\n" + f.name)
    display(Image(filename=str(f), width=950))

In [ ]:
# --- ubicación de todos los artefactos ---
print(f"Carpeta de diagnóstico : {DIR_DIAG.resolve()}")
for n in ["00_RESUMEN_M2.txt", "INFORME_M2.pdf", "metricas_M2.json"]:
    f = DIR_DIAG / n
    print(f"  {'[OK]   ' if f.exists() else '[FALTA]'} {n}")
n_fig = len(list(DIR_DIAG.glob("fig_14*.png")))
n_csv = len(list(DIR_DIAG.glob("*.csv")))
print(f"  {n_fig} figuras PNG y {n_csv} tablas CSV en la misma carpeta.")